## 1. Verify Pre-installed Packages

Check which packages are already available in the environment before installing additional dependencies.

# Neo4j MCP Agent with LangGraph

This notebook demonstrates how to build an AI agent that queries a Neo4j knowledge graph using the **Model Context Protocol (MCP)**. The agent uses LangGraph's ReAct pattern with LangChain MCP adapters to discover and invoke Neo4j tools automatically.

**What you'll do:**
- Connect to a Neo4j MCP Server through an AWS AgentCore Gateway
- Let the agent discover available tools (get-schema, read-cypher)
- Ask natural language questions about SEC financial data
- Observe the agent's schema-first query approach

**Prerequisites:**
- Lab 1 complete (Neo4j Aura instance with SEC data loaded)
- `CONFIG.txt` updated with `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN`

In [ ]:
# Check pre-installed packages
import importlib

packages = [
    "langgraph",
    "langchain_aws",
    "langchain_mcp_adapters",
    "mcp",
    "httpx",
]

for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "installed")
        print(f"  {pkg}: {version}")
    except ImportError:
        print(f"  {pkg}: NOT INSTALLED")

In [ ]:
%pip install -U langgraph langchain-aws langchain-mcp-adapters -q

## 2. Imports

Import the libraries needed for the MCP agent:

- **langchain_aws**: `ChatBedrockConverse` for invoking Claude on Amazon Bedrock
- **langchain_mcp_adapters**: `load_mcp_tools` to convert MCP tools into LangChain tools
- **langgraph**: `create_react_agent` for the ReAct agent loop
- **mcp**: `ClientSession` and `streamablehttp_client` for the MCP transport layer

In [ ]:
import asyncio
import concurrent.futures
import warnings
from datetime import timedelta

from langchain_aws import ChatBedrockConverse
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

warnings.filterwarnings("ignore", category=DeprecationWarning)

print("All imports successful.")

## 3. Configuration

Load settings from `CONFIG.txt`:

- **MODEL_ID**: The Bedrock model to use (e.g., `us.anthropic.claude-sonnet-4-5-20250929-v1:0`)
- **REGION**: AWS region for Bedrock API calls
- **MCP_GATEWAY_URL**: The AgentCore Gateway endpoint for the Neo4j MCP Server
- **MCP_ACCESS_TOKEN**: Authentication token for the gateway

For cross-region inference profiles (MODEL_ID starting with `us.` or `global.`), we also derive a `BASE_MODEL_ID` that some LangChain components require.

In [ ]:
# Load configuration from CONFIG.txt
config = {}
with open("../CONFIG.txt", "r") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            config[key.strip()] = value.strip()

MODEL_ID = config.get("MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
REGION = config.get("REGION", "us-east-1")
GATEWAY_URL = config.get("MCP_GATEWAY_URL", "")
ACCESS_TOKEN = config.get("MCP_ACCESS_TOKEN", "")

# Derive base model ID for cross-region inference profiles
BASE_MODEL_ID = None
for prefix in ["us.", "eu.", "apac.", "global."]:
    if MODEL_ID.startswith(prefix):
        BASE_MODEL_ID = MODEL_ID[len(prefix):]
        break

print(f"Model ID:       {MODEL_ID}")
print(f"Base Model ID:  {BASE_MODEL_ID}")
print(f"Region:         {REGION}")
print(f"Gateway URL:    {GATEWAY_URL[:50]}..." if len(GATEWAY_URL) > 50 else f"Gateway URL:    {GATEWAY_URL}")
print(f"Access Token:   {'[SET]' if ACCESS_TOKEN else '[NOT SET]'}")

# Validate required settings
assert GATEWAY_URL and GATEWAY_URL != "your-gateway-url-here", \
    "MCP_GATEWAY_URL not configured in CONFIG.txt. Update it with your AgentCore Gateway URL."
assert ACCESS_TOKEN and ACCESS_TOKEN != "your-access-token-here", \
    "MCP_ACCESS_TOKEN not configured in CONFIG.txt. Update it with your access token."

## 4. System Prompt

The system prompt instructs the agent to follow a **schema-first approach**:

1. Always retrieve the database schema before writing Cypher queries
2. Use only read-only Cypher (MATCH, RETURN, no CREATE/DELETE/SET)
3. Apply LIMIT clauses to avoid overwhelming results
4. Handle NULL values and missing properties gracefully

The prompt also provides context about the SEC financial dataset so the agent understands the domain.

In [ ]:
SYSTEM_PROMPT = """You are a Neo4j database assistant with access to a knowledge graph 
containing SEC 10-K financial filing data. The graph includes companies, products, 
services, risk factors, financial metrics, executives, and asset manager holdings.

IMPORTANT RULES:

1. SCHEMA FIRST: Always retrieve the database schema before writing any Cypher query.
   Use the schema to understand what node labels, relationship types, and properties 
   exist. Never guess at property names or relationship types.

2. READ-ONLY: Only use read operations (MATCH, RETURN, WITH, WHERE, ORDER BY, LIMIT).
   Never use CREATE, MERGE, SET, DELETE, DETACH DELETE, or any write operations.

3. LIMIT RESULTS: Always include a LIMIT clause (typically LIMIT 25) to avoid 
   returning excessive data. For counting queries, LIMIT is not needed.

4. HANDLE NULLS: Use COALESCE() or WHERE property IS NOT NULL when properties might 
   be missing. Not all nodes have every property.

5. FORMAT CLEARLY: Present results in a readable format. Use tables or lists as 
   appropriate. Cite actual data from the query results.

6. EXPLAIN YOUR WORK: Briefly explain the Cypher query you are running and why, 
   so the user understands how the answer was derived.
"""

print("System prompt configured.")
print(f"Length: {len(SYSTEM_PROMPT)} characters")

## 5. Initialize LLM

Create the Claude model instance using `ChatBedrockConverse`. Temperature is set to 0 for deterministic, factual responses.

In [ ]:
# Build LLM kwargs
llm_kwargs = {
    "model": MODEL_ID,
    "region_name": REGION,
    "temperature": 0,
}

# Add base_model_id for cross-region inference profiles
if BASE_MODEL_ID:
    llm_kwargs["base_model_id"] = BASE_MODEL_ID

llm = ChatBedrockConverse(**llm_kwargs)

print(f"LLM initialized: {MODEL_ID}")

## 6. Query Helper

The query helper manages the complexity of MCP session lifecycle and async execution:

1. **Open an MCP connection** to the gateway using Streamable HTTP with authentication headers
2. **Create a client session** and discover available tools (get-schema, read-cypher)
3. **Build a ReAct agent** with the LLM, discovered tools, and system prompt
4. **Invoke the agent** with the user's question and return the response
5. **Close the connection** when done

The `query()` function wraps this async flow in a synchronous interface for convenient use in notebook cells.

In [ ]:
async def query_async(question: str) -> str:
    """Send a natural language question to the Neo4j MCP agent."""

    # Build auth headers for the gateway
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

    # Open MCP connection via Streamable HTTP
    async with streamablehttp_client(
        url=GATEWAY_URL,
        headers=headers,
        timeout=timedelta(seconds=30),
    ) as (read_stream, write_stream, _):

        # Create MCP client session and initialize it
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            # Discover MCP tools and convert to LangChain tools
            tools = await load_mcp_tools(session)
            print(f"  Discovered {len(tools)} MCP tools: {[t.name for t in tools]}")

            # Create a ReAct agent with the LLM and discovered tools
            agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

            # Invoke the agent with the user's question
            result = await agent.ainvoke({"messages": question})

            # Extract the final response
            return result["messages"][-1].content


def _run_async(coro):
    """Run an async coroutine from a synchronous context (Jupyter-safe)."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(asyncio.run, coro)
        return future.result()


def query(question: str) -> None:
    """Ask a natural language question about the Neo4j database."""
    print(f"Question: {question}\n")
    response = _run_async(query_async(question))
    print(f"\nAnswer:\n{response}")


print("Query helper ready.")

## 7. Demo Queries

Run the following queries to see the agent in action. Watch how it discovers the schema first, then writes Cypher queries based on the actual graph structure.

In [ ]:
query("What is the database schema? Give me a brief summary.")

In [ ]:
query("How many nodes are in the database by label?")

In [ ]:
query("What types of relationships exist in the database?")

## 8. Your Queries

Try your own questions about the SEC financial data. Here are some ideas:

- "How many companies are in the database?"
- "What products does Apple offer?"
- "Which asset managers own stakes in NVIDIA?"
- "What risk factors does Microsoft face?"
- "Show me the financial metrics for Tesla."
- "Who are the executives at Amazon?"
- "Which companies face risk factors related to cybersecurity?"

In [ ]:
query("List 5 sample records from the most populated node type.")

In [ ]:
# query("Your question here")